In [66]:
# --- INSTALLATION ---
# python -m venv .venv
# .venv\Scripts\activate
# pip install ipykernel numpy pandas sqlalchemy pyodbc seaborn matplotlib

<h1> Setting Up

In [67]:
# --- Import Libraries ---
import numpy as np
import pandas as pd

import sqlite3
# from sqlalchemy import create_engine

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

today = pd.Timestamp.today()

# show full columns
pd.set_option('display.max_rows', None)

# embedd plots as static images
%matplotlib inline
# change sizings
matplotlib.rcParams['figure.figsize']=(12,8)


In [68]:
# --- Load Data ---
db_file = 'customer_churn.db'
server = r'(localdb)\MSSQLLocalDB'
database = 'da_projects'
driver = 'ODBC Driver 18 for SQL Server'

# Connect to Database
# connection_string = rf"mssql+pyodbc://@{server}/{database}?driver={driver}&trusted_connection=yes"
# engine = create_engine(connection_string, fast_executemany=True)

# Import All Tables
conn = sqlite3.connect(db_file)
query ="""
        SELECT name 
        FROM sqlite_master 
        WHERE type='table';
        """
tables_df = pd.read_sql(sql=query, con=conn)

for table_name in tables_df['name']:
    if table_name.startswith('sqlite_'):
        continue
    
    # Read Table to Dataframe
    df = pd.read_sql(f"SELECT * FROM {table_name}", conn) 
    # df.to_sql(table_name, engine, if_exists='replace', index=False)
    # print(f'Finish Loading Table {table_name} into {database}')
    
    globals()[f'df_{table_name}'] = df
    print(f'Finish Creating Table df_{table_name}')
    
    # Get Tables Columns
    columns = pd.read_sql(f"PRAGMA table_info({table_name})", conn)
    print(columns['name'].tolist(), '\n')
    
conn.close()

Finish Creating Table df_db_customer
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode'] 

Finish Creating Table df_db_subscription
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score'] 

Finish Creating Table df_db_support
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment'] 



In [69]:
display(df_db_customer.info())
display(df_db_customer.head(3))
display(df_db_subscription.info())
display(df_db_subscription.head(3))
display(df_db_support.info())
display(df_db_support.head(3))

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerid  21 non-null     str   
 1   name        21 non-null     str   
 2   country     18 non-null     str   
 3   state       21 non-null     str   
 4   gender      21 non-null     str   
 5   dob         21 non-null     str   
 6   interests   4 non-null      str   
 7   pincode     0 non-null      object
dtypes: object(1), str(7)
memory usage: 1.4+ KB


None

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,NaN,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None


<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               21 non-null     str    
 1   subscription_start_date  21 non-null     str    
 2   subscription_type        21 non-null     str    
 3   renewal_date             21 non-null     str    
 4   plan_type                21 non-null     str    
 5   contract_type            21 non-null     str    
 6   cancellation_date        6 non-null      str    
 7   cancellation_reason      6 non-null      str    
 8   monthly_charges          21 non-null     float64
 9   cltv                     21 non-null     int64  
 10  churn_score              21 non-null     int64  
dtypes: float64(1), int64(2), str(8)
memory usage: 1.9 KB


None

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaN,NaN,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaN,NaN,6.99,210,34


<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customerid      9 non-null      str   
 1   complaint_date  9 non-null      str   
 2   escalations     9 non-null      str   
 3   csat_score      9 non-null      int64 
 4   col_1           0 non-null      object
 5   comment         4 non-null      str   
dtypes: int64(1), object(1), str(4)
memory usage: 564.0+ bytes


None

,customerid,complaint_date,escalations,csat_score,col_1,comment
0,0003-MKNFE,2024-08-28 00:00:00,N,60,None,service issue
1,0003-MKNFE,2024-08-28 00:00:00,Y,10,None,demaned refund
2,0013-EXCHZ,2024-01-20 00:00:00,Y,20,None,NaN


<h1> Data Cleaning

<h2> Rename Columns

In [70]:
for table_name in [df_db_customer, df_db_subscription, df_db_support]:
    table_name.rename(columns={'customerid':'customer_id'}, inplace=True)

<h2> Drop Columns

In [71]:
# df_db_customer: drop column interests and pincode
df_db_customer = df_db_customer.drop(columns=['interests', 'pincode'])

# df_db_support: drop column col_1, comment
df_db_support = df_db_support.drop(columns=['col_1', 'comment'])

<h2> Change Data Type

In [72]:
# df_db_customer: change column dob to datetime
df_db_customer['dob'] = pd.to_datetime(df_db_customer['dob'])

# df_db_subscription: change column subscription_start_date, renewal_date, cancellation_date to date time
for col in ['subscription_start_date', 'renewal_date', 'cancellation_date']:
    df_db_subscription[col] = pd.to_datetime(df_db_subscription[col])
 
# df_db_support: change column complaint_date to datetime
df_db_support['complaint_date'] = pd.to_datetime(df_db_support['complaint_date'])

<h2> Data Standardization

In [73]:
# df_db_customer: column gender
df_db_customer['gender'] = df_db_customer['gender'].replace({'Men' : 'Male', 'Women' : 'Female'})
df_db_customer['gender'].unique()

<StringArray>
['Male', 'Female']
Length: 2, dtype: str

<h2> Fix Missing Value

In [74]:
# df_db_customer: column country
country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()
df_db_customer['country'] = df_db_customer['country'].fillna(country_mapping)

<h1> Feature Engineering and Data Analysis

In [75]:
display(df_db_customer.head(3))
display(df_db_subscription.head(3))
display(df_db_support.head(3))

,customer_id,name,country,state,gender,dob
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15


,customer_id,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,NaN,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,NaN,6.99,210,34


,customer_id,complaint_date,escalations,csat_score
0,0003-MKNFE,2024-08-28,N,60
1,0003-MKNFE,2024-08-28,Y,10
2,0013-EXCHZ,2024-01-20,Y,20


<h2> Create New Columns

In [76]:
# --- df_db_customer ---
df_db_customer['age'] = (today - df_db_customer['dob']).dt.days // 365

age_group_mapping = [18, 30, 40, 50, 60, 100]
age_group_labels = ['18-29', '30-39', '40-49', '50-59', '60+']
df_db_customer['age_group'] = pd.cut(df_db_customer['age'], bins=age_group_mapping, labels=age_group_labels, right=False)

# --- df_db_subscription ---
df_db_subscription['churn_flag'] = np.where(df_db_subscription['cancellation_date'].notna(), 1, 0)

churn_risk_mapping = [0, 50, 70, 100]
churn_risk_labels = ['low', 'medium', 'high']
df_db_subscription['churn_risk'] = pd.cut(df_db_subscription['churn_score'], bins=churn_risk_mapping, labels=churn_risk_labels, right=False)

# --- df_db_support ---
df_db_support['complaint_count'] = df_db_support.groupby('customer_id')['customer_id'].transform('count')
# Keep only the last complaint since there's total complaint already
df_db_support = df_db_support.sort_values('complaint_date').drop_duplicates('customer_id', keep='last')

<h2> Join Dataframes

In [77]:
df = (df_db_customer
        .merge(df_db_subscription, on='customer_id', how='left')
        .merge(df_db_support, on='customer_id', how='left'))
df.head()

,customer_id,name,country,state,gender,dob,age,age_group,subscription_start_date,subscription_type,...,cancellation_reason,monthly_charges,cltv,churn_score,churn_flag,churn_risk,complaint_date,escalations,csat_score,complaint_count
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12,44,40-49,2021-03-15,Refferal,...,NaN,13.99,627,12,0,low,NaT,NaN,NaN,NaN
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23,30,30-39,2020-08-01,Paid,...,Switched to competitor,12.99,1150,91,1,high,2024-08-28,Y,10.0,2.0
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15,48,40-49,2022-11-20,Organic,...,NaN,6.99,210,34,0,low,NaT,NaN,NaN,NaN
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30,25,18-29,2019-05-10,Paid,...,NaN,22.99,1725,8,0,low,NaT,NaN,NaN,NaN
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05,36,30-39,2023-01-05,Refferal,...,Too expensive,13.99,195,88,1,high,2024-01-20,Y,20.0,1.0


<h2> Export Data

In [78]:
df.to_csv('cleaned_churn_data.csv')

<h2> Data Analysis

In [79]:
# Churn Rate
churn_rate = df['churn_flag'].mean() * 100
print(f'Churn Rate: {round(churn_rate, 2)}%')

Churn Rate: 28.57%


In [80]:
# Retention Rate
retention_rate = 100 - churn_rate
print(f'Retention Rate: {round(retention_rate, 2)}%')

Retention Rate: 71.43%


In [81]:
# Churn by Plan Type
churn_by_plan = df.groupby('plan_type')['churn_flag'].mean().mul(100).round(2).reset_index(name='churn_rate_percentage')
display(churn_by_plan)

,plan_type,churn_rate_percentage
0,Basic,60.00
1,Premium,14.29
2,Standard,22.22


In [82]:
# Churn, Total Revenue, Total Users by State
data_by_state = (df.groupby('state')[['churn_flag', 'monthly_charges', 'customer_id']]
                    .agg(
                        churn_rate_percentage=('churn_flag', 'mean'), 
                        total_revenue=('monthly_charges', 'sum'), 
                        total_users=('customer_id', 'count'))
                    .reset_index())
data_by_state['churn_rate_percentage'] = data_by_state['churn_rate_percentage'].mul(100).round(2)
display(data_by_state)

,state,churn_rate_percentage,total_revenue,total_users
0,Delhi,25.00,52.96,4
1,Karnataka,100.00,20.98,2
2,Kathmandu,0.00,20.98,2
3,Maharashtra,0.00,50.97,3
4,Meghalaya,66.67,42.97,3
5,Nagaland,0.00,22.99,1
6,Rajasthan,0.00,36.98,2
7,Telangana,50.00,30.98,2
8,Uttar Pradesh,0.00,115.98,2


In [83]:
# Churn, Total Revenue, Total Users by Subscription
data_by_subscription = (df.groupby('subscription_type')[['churn_flag', 'monthly_charges', 'customer_id']]
                    .agg(
                        churn_rate_percentage=('churn_flag', 'mean'), 
                        total_revenue=('monthly_charges', 'sum'), 
                        total_users=('customer_id', 'count'))
                    .reset_index())
data_by_subscription['churn_rate_percentage'] = data_by_subscription['churn_rate_percentage'].mul(100).round(2)
display(data_by_subscription)

,subscription_type,churn_rate_percentage,total_revenue,total_users
0,Organic,0.00,145.91,9
1,Paid,16.67,174.94,6
2,Refferal,83.33,74.94,6


In [84]:
# Average Revenue per User
arpu = df['monthly_charges'].mean()
print(f'Average Revenue per User: {round(arpu, 2)}')

Average Revenue per User: 18.85


In [85]:
# Average Customer Age
avg_age = df['age'].mean()
print(f'Average Customer Age: {round(avg_age)}')

# Churn, Total Revenue, Total Users by Groups of Age
data_by_age = (df.groupby('age_group')[['churn_flag', 'monthly_charges', 'customer_id']]
                    .agg(
                        churn_rate_percentage=('churn_flag', 'mean'), 
                        total_revenue=('monthly_charges', 'sum'), 
                        total_users=('customer_id', 'count'))
                    .reset_index())
data_by_age['churn_rate_percentage'] = data_by_age['churn_rate_percentage'].mul(100).round(2)
display(data_by_age)

Average Customer Age: 37


,age_group,churn_rate_percentage,total_revenue,total_users
0,18-29,0.00,96.95,5
1,30-39,42.86,108.93,7
2,40-49,25.00,180.92,8
3,50-59,100.00,8.99,1


In [86]:
# Average Customer Tenure (Time Customer Have Subscription)
df['tenure_days'] = np.where(
    df['cancellation_date'].isna(),
    (today - df['subscription_start_date']).dt.days,
    (df['cancellation_date'] - df['subscription_start_date']).dt.days
    )
avg_tenure = df['tenure_days'].mean()
print(f'Average Customer Tenure: {round(avg_tenure)} days.')

Average Customer Tenure: 1548 days.


In [87]:
# Revenue At Risk
risk_revenue = df[df['churn_flag'] == 1]['monthly_charges'].sum()
print(f'Revenue At Risk: {round(risk_revenue, 2)}')

Revenue At Risk: 73.94


In [88]:
# Escalation Rate
escalation_rate = (df['escalations'] == 'Y').mean() * 100
print(f'Escalation Rate: {round(escalation_rate, 2)}%')

Escalation Rate: 19.05%


In [89]:
# Average Complaint per User
avg_complaint = df['complaint_count'].sum() / df['customer_id'].nunique()
print(f'Average Complaint per User: {round(avg_complaint, 2)}')

Average Complaint per User: 0.43


In [90]:
# Correlation between Escalation vs Churn
escalations = pd.Series(np.where(df['escalations'] == 'Y', 1, 0)) 
corr = escalations.corr(df['churn_score'])
print(f'Correlation between Escalation vs Churn: {round(corr, 2)}')

Correlation between Escalation vs Churn: 0.63


<h1> Visualization

<h2> Visualization using Matplotlib

<h2> Visualization using Seaborn

<h2> Pivot Table